<table>
  <tr>
    <td><div align="left"><font size="30">Exploring images and pixels</font></div></td>
    <td><img src="https://github.com/petercorke/machinevision-toolbox-python/raw/main/docs/figs/VisionToolboxLogo_NoBackgnd@2x.png?raw=1" width="200" height="158" style="height:auto"></td>
  </tr>
</table>

(c) Peter Corke 2025

Robotics, Vision & Control: Python, see Section 11.1

## Configuring the Jupyter environment
We need to import some packages to help us with linear algebra (`numpy`), graphics (`matplotlib`), and machine vision (`machinevisiontoolbox`). If you're running locally you need to have these packages installed.

<div class="alert alert-info">
If you're running on CoLab or in the browser we have to first install machinevisiontoolbox, which is not preinstalled, so the next cell will be slow.
</div>

Set matplotlib for interactive display within the notebook, then import the `Image` object from the Machine Vision Toolbox.

In [ ]:
# MVTB_BOOTSTRAP_CELL: installs the package, which takes a short time, so wait for the line "Running ..." below.  Generated by sync_bootstrap.py from _mvtb_nb_bootstrap.py, do not edit

import subprocess
import sys
PACKAGE = 'machinevision-toolbox-python'
IMPORT_NAME = 'machinevisiontoolbox'
GIT_URL: str | None = 'https://github.com/petercorke/machinevision-toolbox-python'
JL_PACKAGES = ['numpy', 'scipy', 'matplotlib', 'ipympl', 'pillow', 'typing_extensions', 'opencv-python', 'ansitable', 'mvtb-data', 'tqdm', 'requests', 'ipywidgets', 'spatialmath-python', 'pgraph-python']
JL_NO_DEPS: list[str] = []
JL_PACKAGE_NO_DEPS = True
JL_EXTRAS: dict[str, list[str]] = {}
COLAB_PACKAGES: list[str] = []

def _version_of(package: str, module) -> str:
    import importlib.metadata
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return str(getattr(module, '__version__', 'unknown'))

def _older(version: str, required: str) -> bool:
    try:
        from packaging.version import InvalidVersion, Version
        return Version(version) < Version(required)
    except (ImportError, InvalidVersion):
        return False

async def ensure_installed(extras: tuple[str, ...] | list[str]=(), packages: tuple[str, ...] | list[str]=(), min_version: str | None=None) -> bool:
    if sys.platform == 'emscripten':
        try:
            import piplite
            install = piplite.install
        except ImportError:
            import micropip
            install = micropip.install
        print(f'Installing {PACKAGE}, this takes a short time...', flush=True)
        for name in JL_NO_DEPS:
            await install(name, deps=False)
        for name in JL_PACKAGES:
            await install(name)
        for extra in extras:
            for name in JL_EXTRAS.get(extra, []):
                await install(name)
        for name in packages:
            await install(name)
        await install(PACKAGE, deps=not JL_PACKAGE_NO_DEPS)
        where, colab = ('in browser', False)
    else:
        try:
            import google.colab
        except ImportError:
            where, colab = ('locally', False)
        else:
            print(f'Installing {PACKAGE}, this takes a short time...', flush=True)
            requirement = PACKAGE
            if extras:
                requirement += '[' + ','.join(extras) + ']'
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', requirement] + list(packages) + COLAB_PACKAGES, check=True)
            try:
                from matplotlib.backends import backend_registry
            except ImportError:
                pass
            else:
                clear = getattr(backend_registry, '_clear', None)
                if clear is not None:
                    clear()
            from google.colab import output
            output.enable_custom_widget_manager()
            where, colab = ('on Colab', True)
    import importlib
    module = importlib.import_module(IMPORT_NAME)
    version = _version_of(PACKAGE, module)
    if colab and min_version is not None and _older(version, min_version) and GIT_URL:
        print(f'{PACKAGE} {version} on PyPI is older than {min_version}, installing from GitHub...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', f'{PACKAGE} @ git+{GIT_URL}@main'], check=True)
        importlib.invalidate_caches()
        version = _version_of(PACKAGE, module)
        where += ', development version from GitHub'
    if not colab and where == 'locally':
        import importlib.metadata
        for name in packages:
            try:
                importlib.metadata.version(name)
            except importlib.metadata.PackageNotFoundError:
                print(f'This notebook needs {name}: pip install {name}')
    if min_version is not None and where in ('locally', 'on Colab') and _older(version, min_version):
        print(f'This notebook needs {PACKAGE} {min_version} or later: pip install -U {PACKAGE}')
    print(f'Running {where} using {PACKAGE} v{version}')
    return colab

COLAB = await ensure_installed()


In [ ]:
%matplotlib ipympl
import matplotlib.pyplot as plt
plt.close("all")   # re-running the notebook from the top starts with no figures
from machinevisiontoolbox import Image

# The `Image` class

To illustrate the usage of the `Image` class we will import a colorful image  

In [ ]:
flowers = Image.Read("flowers4.png", mono=False)

The `mono` option means don't convert it to greyscale, though after you've worked through the notebook, it would be interesting to rerun with this option set to `True`.

The result is an object of type

In [ ]:
type(flowers)

that contains the color image and has many methods to operate on the image.

A really useful method is `disp()` which will display the image and allow interactive browsing of the pixel values.  The `disp()` methods has lots of options &ndash; <strong>check them out.</strong>

In [ ]:
flowers.disp()

As you move the cursor around, the cursor location and the pixel values are updated beneath the image.  Note that the pixel value is a 3-vector whose elements are the amount of red (R), green (G) and blue (B) at that point.  

<div class="alert alert-success">
Move the cursor over the red, white, yellow and blue (very small, see if you can find it) flowers, the green leaves, and the dark shadows &ndash; do the RGB values make sense?
</div>

<div class="alert alert-success">
Explore also the buttons to the left of the image.  They allow you to zoom in to a region, pan and scroll that zoomed view, or save the view to a file.
</div>

The size of the image is usually described in terms of width x height

In [ ]:
flowers.size

and the individual pixel values have a datatype

In [ ]:
flowers.dtype

which in this case in unsigned 8-bit integer.  Their range can be determined by

In [ ]:
print(f"pixel values can span the range from {flowers.minval} to {flowers.maxval}")

The image comprises

In [ ]:
flowers.nplanes

"planes" that hold the red, green and blue values respectively.

All of this metadata about the image can be seen by displaying the value of the `Image` instance, 

In [ ]:
flowers

you don't see all the pixel values, just the key properties.  Many other properties can be accessed as attributes or properties of the instance.

# Color planes

A color image can be considered as a 3D stack of images, each layer in the stack is called a plane.
Color images typically have a red, green and blue plane.  The green plane shows the amount of green at every point in the original image.



In [ ]:
green = flowers["G"]
green

Note that the resulting "green plane" is another `Image` instance, but this time it has only

In [ ]:
green.nplanes

plane, it is a greyscale image.  It is easy to display

In [ ]:
green.disp()

and we see that the white flowers appear bright, they have a strong green component, as do the leaves, but the red flowers appear very dark, they have little green.

The `Image` object supports an arbitrary number of planes in any order.  The `colororder` dict 

In [ ]:
flowers.colororder

is a mapping from colorplane name to the plane index.  The example above we could also have done as

In [ ]:
green = flowers[1]

since the green plane, in this case has a plane index of 1. However the RGB order is just a convention, and we shouldn't rely on assuming that the green plane is always index 1.

The actual distribution of the pixel values in each plane is

In [ ]:
flowers.stats

which shows that the red, green and blue values all vary from 0 to 255, the maximum permissible range.  This is unusual, but the white flowers have pushed each of the color channels to the limit.

But where are all the zero pixel values?  We can easily find the pixels where R==0 by

In [ ]:
(flowers["R"] == 0).disp()

the comparison creates an `Image` instance with boolean values, true where the pixel value is zero.  The `disp()` method displays `True` as white, and we can see a few white pixels in the lower left of the image.

<div class="alert alert-success">
Modify this to find where zero-valued green and blue pixels are. Or where the pixels have a blue component less than 50.
</div>

The value at a particular coordinate (280,220) in the image is given by

In [ ]:
flowers[280,220]

and we see the result is a 3-vector, as a 1d Numpy array.  This is the amount of red, green and blue at that particular point.  Red dominates in this case because the point belongs to one of the red flowers.

We can also grab a subregion of the image using Python slice notation

In [ ]:
sub = flowers[480:540,330:390]
sub

and the result is now another `Image` instance which we can display

In [ ]:
sub.disp()

and it is the elusive blue flower!

The pixel data is actually held in a Numpy array encapsulated by the `Image` object.  We can access it

In [ ]:
pixdata = flowers.array
type(pixdata)

and see that it is indeed a Numpy array.  Its shape

In [ ]:
pixdata.shape

is given as the number of rows (height), columns (width) and planes.  Compare these numbers with the properties of the `Image` instance shown above.

We can access the same red flower pixel as above by

In [ ]:
pixdata[220,280]

but note that we have had to swap the indices.  Numpy coordinates are given in column, row order, whereas the `Image` instance supports the image coordinate system convention of column then row.

<div class="alert alert-success">
Repeat this notebook, with <code>mono</code> set to <code>True</code> in the second cell.  Note the differences.
</div>